In [ ]:
!pip install -U bitsandbytes
#!pip install -q ninja packaging
#!pip install flash-attn --no-build-isolation

In [ ]:
import torch
import pickle
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import re
import torch
from datasets import load_dataset
from collections import Counter


In [ ]:
device = "cuda"

## LOAD JUDGE MODEL

In [ ]:
model_id = "AtlaAI/Selene-1-Mini-Llama-3.1-8B"

In [ ]:
# Configuration for 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # A100/L4 support bfloat16
)

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    #attn_implementation="flash_attention_2",
    dtype=torch.bfloat16
)

In [ ]:
tokenizer.model_max_length = 200000
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.eos_token_id

**Change to false for similarity based examples:**

In [ ]:
use_cluster_examples = False

**Change to false for test set calculation:**

In [ ]:
use_validation = False

Load the correct files:

In [ ]:
if use_validation:
  with open("INSTANCES_100_VAL.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_300_VAL_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
else:
  with open("INSTANCES_200_TEST.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_600_TEST_SIM.pkl", "rb") as f:
      examples = pickle.load(f)

In [ ]:
# Pre-group examples by instance ID (3 examples per ID)
# dictionary with keys = instance ID and value = list filled with 3 dictionaries of examples belonging to instance ID

examples_map = {}
for ex in examples:
    iid = ex['inst_id']
    if iid not in examples_map:
        examples_map[iid] = []
    examples_map[iid].append(ex)

# PHASE 1 - BASELINE ACCURACY

Define Functions:

In [ ]:
# NO EXPLANATIONS
def format_demonstrations_no_expl(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
def extract_choice(response):
    # Pattern 1: Strict "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-B])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Pattern 2: Loose "Result: A"
    match = re.search(r"Result:\s*([A-B])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    return "FAIL"

In [ ]:
def instruction_prompt(examples, test_input):
  instruction_str = ""
  start = "Your goal is to select the summary from candidates A and B that was produced by the Target Model for a news article. You will be provided with 3 examples of the Target Model's behavior. For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.\n"
  instruction_str += start
  instruction_str += "Here are the examples:\n"
  instruction_str += examples + "\n"
  instruction_str += f"Here is the new Article: {test_input}\n"
  return instruction_str

Define Prompt:

In [ ]:
prompt_1 = """
  You are an Expert LLM Evaluator specializing in Model Identification.
  Your goal is to select the response that matches the Target Model described in the instruction.
  Select Response A or Response B, whichever is more consistent with the Target Model demonstrations provided in the instruction.

  Here are some rules of the evaluation:
  (1) You should prioritize evaluating whether the response is consistent with the examples provided in the instruction.
  (2) You should NOT select a response because it is more fluent, detailed, or human-like.
  (3) You should avoid any potential bias and your judgment should be as objective as possible.
  Here are some potential sources of bias:
  - The order in which the responses were presented should NOT affect your judgment,
  as Response A and Response B are **equally likely** to be created by the Target Model.
  - The length of the responses should only be a factor if it matches the Target Model behavior observable in the given examples.

  Your reply should strictly follow this format:
  **Reasoning:** <feedback evaluating the responses>

  **Result:** <A or B>

  Here is the data.

  Instruction:
  ```
  {instruction}
  ```

  Response A:
  ```
  {Candidate1}
  ```

  Response B:
  ```
  {Candidate2}
  ```

  """

In [ ]:
# Dictionary to store final accuracies: {'distilbart_vs_llama': 0.85}
#all_results_dict = {}

In [ ]:
results_ph1 = {}
score_counter_ph1 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx
    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        #("gold", test_instance["summaries"]["gold"]),
        #("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        #("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_no_expl(current_examples)
    instruction = instruction_prompt(demos, test_instance["input"])

    final_prompt = prompt_1.format(
        instruction=instruction,
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
    )

    print(data_idx)
    print(final_prompt)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph1[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph1[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "predicted_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

In [ ]:
#gold_acc = sum(1 for result in results_ph1.values() if result["predicted_model"] == "gold") / len(results_ph1)
#flan_acc = sum(1 for result in results_ph1.values() if result["predicted_model"] == "flan") / len(results_ph1)

#all_results_dict[f"{candidates[0][0]} vs {candidates[1][0]}"] = f"Gold Acc: {gold_acc:.2%}, Flan Acc: {flan_acc:.2%}"

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT PHASE 1 PAIRWISE")
print("Distribution:")
for model_sum, count in score_counter_ph1.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph1['UNKNOWN'] > 0:
    print(f"\nWARNING: {score_counter_ph1['UNKNOWN']} responses could not be parsed.")

#PHASE 2

Define Functions:

In [ ]:
# TOP SENTENCE
def format_demonstrations_top(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentence: {ex['top_sentence']['sentence']}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
def instruction_prompt(examples, test_input):
  instruction_str = ""
  start = "Your goal is to select the summary from candidates A and B that was produced by the Target Model for a news article. You will be provided with 3 examples of the Target Model's behavior. For each example you will also be provided with the specific sentence from the article that the Target Model prioritized to create its summary. For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.\n"
  instruction_str += start
  instruction_str += "Here are the examples:\n"
  instruction_str += examples + "\n"
  instruction_str += f"Here is the new Article: {test_input}\n"
  return instruction_str

In [ ]:
def extract_choice(response):
    # Pattern 1: Strict "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-B])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Pattern 2: Loose "Result: A"
    match = re.search(r"Result:\s*([A-B])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    return "FAIL"

Define Prompt:

In [ ]:
prompt_2_top = """
  You are an Expert LLM Evaluator specializing in Model Identification.
  Your goal is to select the response that matches the Target Model described in the instruction.
  Select Response A or Response B, whichever is more consistent with the Target Model demonstrations provided in the instruction.

  Here are some rules of the evaluation:
  (1) You should prioritize evaluating whether the response is consistent with the examples provided in the instruction.
  (2) You should NOT select a response because it is more fluent, detailed, or human-like.
  (3) You should avoid any potential bias and your judgment should be as objective as possible.
  Here are some potential sources of bias:
  - The order in which the responses were presented should NOT affect your judgment,
  as Response A and Response B are **equally likely** to be created by the Target Model.
  - The length of the responses should only be a factor if it matches the Target Model behavior observable in the given examples.

  Your reply should strictly follow this format:
  **Reasoning:** <feedback evaluating the responses>

  **Result:** <A or B>

  Here is the data.

  Instruction:
  ```
  {instruction}
  ```

  Response A:
  ```
  {Candidate1}
  ```

  Response B:
  ```
  {Candidate2}
  ```

  """

In [ ]:
results_ph2 = {}
score_counter_p2 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx
    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    # select two models
    candidates = [
        #("gold", test_instance["summaries"]["gold"]),
        #("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        #("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top(current_examples)
    instruction = instruction_prompt(demos, test_instance["input"])

    final_prompt = prompt_2_top.format(
        instruction=instruction,
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
    )

    print(data_idx)
    print(final_prompt)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()


    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_p2[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "predicted_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT PHASE 2 PAIRWISE")
print("Distribution:")
for model_sum, count in score_counter_p2.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_p2['UNKNOWN'] > 0:
    print(f"\nWARNING: {score_counter_p2['UNKNOWN']} responses could not be parsed.")